In [1]:
import torch
import open_clip
from PIL import Image
import chromadb
import os
from pathlib import Path

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)

model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [3]:
!nvidia-smi

Mon Feb 23 23:51:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P8              1W /   50W |    3873MiB /   4096MiB |     21%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
def generate_image_embedding(image_path):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model.encode_image(image)

    features = features / features.norm(dim=-1, keepdim=True)

    return features.cpu().numpy()[0]

In [6]:
path = Path("data/video_generation/embeddings/chroma_db_landmarks")
client = chromadb.PersistentClient(path=path)
collection = client.create_collection(name="landmarks_images",get_or_create=True)

In [ ]:
image_folder = Path("data/video_generation/raw/landmarks_images")

image_files = list(image_folder.rglob("*.jpg")) + list(image_folder.rglob("*.png")) + list(image_folder.rglob("*.jpeg"))
image_id = 0
for img_path in image_files:
    emb = generate_image_embedding(str(img_path))
    
    landmark_name = img_path.parent.name
    file_name = img_path.name

    collection.add(
        ids=[str(image_id)],  
        embeddings=[emb],
        metadatas=[{"landmark": landmark_name,
                    "is_builder":"yes" if "builder" in file_name.lower() else "no",
                    "is_plan":"yes" if "plan" in file_name.lower() else "no",
                    "is_location":"yes" if "location" in file_name.lower() else "no"}],
        documents=[str(img_path)]
    )
    image_id+=1

In [8]:
print(collection.count())

0


In [ ]:
results = collection.get(limit=11)
for meta, doc in zip(results["metadatas"], results["documents"]):
    print(f"{doc} → {meta}")

Final_video_landmarks\Al Qurn\1.jpg → {'is_location': 'no', 'is_builder': 'no', 'is_plan': 'no', 'landmark': 'Al Qurn'}
Final_video_landmarks\Al Qurn\10.jpg → {'is_plan': 'no', 'is_location': 'no', 'landmark': 'Al Qurn', 'is_builder': 'no'}
Final_video_landmarks\Al Qurn\3.jpg → {'landmark': 'Al Qurn', 'is_location': 'no', 'is_plan': 'no', 'is_builder': 'no'}
Final_video_landmarks\Al Qurn\5.jpg → {'is_builder': 'no', 'is_plan': 'no', 'landmark': 'Al Qurn', 'is_location': 'no'}
Final_video_landmarks\Al Qurn\6.jpg → {'is_builder': 'no', 'is_plan': 'no', 'landmark': 'Al Qurn', 'is_location': 'no'}
Final_video_landmarks\Al Qurn\Al Qurn_20.jpg → {'landmark': 'Al Qurn', 'is_location': 'no', 'is_builder': 'no', 'is_plan': 'no'}
Final_video_landmarks\Al Qurn\Al Qurn_21.jpg → {'is_location': 'no', 'is_plan': 'no', 'landmark': 'Al Qurn', 'is_builder': 'no'}
Final_video_landmarks\Al Qurn\Al Qurn_24.jpg → {'is_plan': 'no', 'is_builder': 'no', 'is_location': 'no', 'landmark': 'Al Qurn'}
Final_video_

In [9]:
results = collection.get(include=["embeddings", "metadatas", "documents"], limit=2)
embeddings = results["embeddings"]
print(len(embeddings[0]))


1024
